# 06 — Silver Authentication Events

**Input:** `Capstone_team01.bronze.device_authentication_events`  
**Outputs:** `Capstone_team01.silver.device_authentication_events` and its quarantine table.

Run `08_silver_master_data` and `05_silver_transactions` first.

Rules:
- `auth_event_id` is required and unique.
- `event_ts` must parse as a timestamp when present.
- `attempt_number` must parse as an integer when present.
- Referenced transaction and customer must exist in Silver.
- Duplicate event IDs are quarantined in full.
- No undocumented allowed-value list is imposed on authentication result, method, or risk signal.


In [ ]:
from pyspark.sql import functions
from pyspark.sql.window import Window

spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("Capstone_team01.bronze.device_authentication_events")

df = (df
      .withColumn("_raw_event_ts", functions.col("event_ts"))
      .withColumn("_raw_attempt_number", functions.col("attempt_number"))
      .withColumn("event_ts", functions.to_timestamp("event_ts"))
      .withColumn("attempt_number", functions.col("attempt_number").cast("integer")))

duplicate_window = Window.partitionBy("auth_event_id")
df = df.withColumn("_key_count", functions.count("*").over(duplicate_window))

df = df.withColumn(
    "_dq_reason",
    functions.concat_ws(
        ";",
        functions.when(
            functions.col("auth_event_id").isNull() |
            (functions.trim(functions.col("auth_event_id")) == ""),
            "MISSING_AUTH_EVENT_ID"
        ),
        functions.when(functions.col("_key_count") > 1, "DUPLICATE_AUTH_EVENT_ID"),
        functions.when(functions.col("_raw_event_ts").isNotNull() & functions.col("event_ts").isNull(), "INVALID_EVENT_TS"),
        functions.when(functions.col("_raw_attempt_number").isNotNull() & functions.col("attempt_number").isNull(), "INVALID_ATTEMPT_NUMBER")
    )
)

transactions = spark.table("Capstone_team01.silver.card_transactions").select("transaction_id").distinct()
customers = spark.table("Capstone_team01.silver.customer_master").select("customer_id").distinct()

df = (df
      .join(transactions.withColumn("_transaction_found", functions.lit(True)), "transaction_id", "left")
      .join(customers.withColumn("_customer_found", functions.lit(True)), "customer_id", "left"))

df = df.withColumn(
    "_dq_reason",
    functions.concat_ws(
        ";", functions.col("_dq_reason"),
        functions.when(functions.col("_transaction_found").isNull(), "UNKNOWN_TRANSACTION_ID"),
        functions.when(functions.col("_customer_found").isNull(), "UNKNOWN_CUSTOMER_ID")
    )
).withColumn("_quarantined_at", functions.current_timestamp())

quarantine_df = df.filter(functions.col("_dq_reason") != "")
clean_df = df.filter(functions.col("_dq_reason") == "")

clean_df = clean_df.drop(
    "_raw_event_ts", "_raw_attempt_number", "_key_count", "_dq_reason",
    "_quarantined_at", "_transaction_found", "_customer_found"
)

(
    clean_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.device_authentication_events")
)
(
    quarantine_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.quarantine_device_authentication_events")
)

print("Clean authentication events:", clean_df.count())
print("Quarantined authentication events:", quarantine_df.count())
display(clean_df.limit(10))
